# 05_5 — Narrador Natural ROBUSTO (viabilidade de pico de corridas no SBPA)

Versão reescrita e mais robusta do narrador. Responde à pergunta do **motorista**:
*"Vale a pena ir ao Aeroporto Salgado Filho no dia X, às Y horas, esperando um pico de
corridas?"* — sendo o pico originado **principalmente por atrasos de voos e por eventos
climáticos**.

## Como funciona (fluxo dos 6 passos)
1. **Motorista informa dia + horário** (linguagem natural, ex.: *"amanhã às 18h"*). A hora é
   interpretada em **horário local de Porto Alegre** (`America/Sao_Paulo`).
2. O **pico de corridas** é modelado por duas dimensões: **atraso de voos** e **eventos
   climáticos** (os dois modelos pareados exportados pelo `05_6`).
3. Buscamos, para o dia/hora pedido: **condições meteorológicas** via **Open-Meteo** (previsão
   futura) e **contexto de atrasos de voos** via **perfil histórico** (por hora × dia-da-semana)
   do CSV `03_voos_atrasados_sbpa.csv`.
4. Com essas informações, montamos as features **no mesmo espaço/namespace e em UTC** que os
   modelos aprenderam e verificamos se as condições **condizem com as que geraram corridas** no
   passado (probabilidade de cada modelo pareado).
5. **Se houver demanda** (probabilidade ≥ limiar): dizemos **SIM** e detalhamos **quais features
   e quais valores** puxaram a decisão — numa *ficha técnica* + narração em linguagem natural.
   (O motorista poderá conferir esses valores pessoalmente e reportar se a previsão acertou.)
6. **Se não houver demanda** no horário pedido: varremos uma janela de **+2 horas para frente**
   (+1h, +2h) e informamos se há possibilidade em algum horário próximo.

## Contrato de entrada (do `05_6`)
`joblib.load` de `modelo_par_clima.joblib` / `modelo_par_atraso.joblib` + `*_meta.json`
(`janela`, `dist_aeroporto`, `features_lgbm`, `macro_f1_bal`, `macro_f1_nat`, `pr_auc`,
`posfato_atraso`). As features são reconstruídas **exatamente** com esses nomes, em **UTC** e
janela **1h**.

> **Pré-requisito exato:** os modelos consumidos aqui são os exportados por
> **`05_6_FeatureEng_Individual_vs_Agrupado.ipynb`** (a versão de PRODUÇÃO, seção de
> `treinar_e_exportar_par`). NÃO confundir com o antigo `05_6_Analise_Pareada_Clima_Atraso`,
> cujo pré-processamento (fillna global) é incompatível com o tratamento de ausentes abaixo.

## Rigor: consistência treino×inferência (train/serve)
- **Fuso:** o `05_6` derivou `hora/mes/dia_semana` e o clima em **UTC**. Aqui convertemos a hora
  local do motorista para UTC ANTES de montar as features (senão haveria erro de ~3h nos
  atributos temporais).
- **Ausentes (NaN):** no `05_6_FeatureEng` a base de produção preenche com 0 **apenas** as
  features de voo (janela sem voo atrasado); as colunas climáticas e `lag/diff` permanecem **NaN**
  quando não há leitura, e `preparar_par_ts`/`treinar_lgbm_par` treinam **sem** `fillna` global —
  ou seja, o LightGBM aprende a direção do *split* de ausência do clima. Este narrador replica
  isso em `_preparar_X` (clima → NaN; voos → 0) para não haver *train/serve skew*.
- **Vocabulário climático:** o narrador só consegue repor via Open-Meteo as variáveis físicas
  cruas; índices UTCI/compostos são **excluídos** do treino pelo `05_6` (`eh_score_composto` casa
  `utci`), então não entram em `features_lgbm`. Um diagnóstico na seção 5 sinaliza qualquer feature
  do modelo que a Open-Meteo não reproduza (serviria NaN constante).
- **Sinal de voo marginal:** o perfil histórico de voos (seção 4) é a média **marginal** por
  hora×dia-da-semana incluindo as janelas SEM atraso como 0 — reproduzindo a distribuição que o
  treino viu (não a média condicional, que superestimaria o atraso típico).


## 1. Setup


In [ ]:
!pip install -q anthropic lightgbm shap h3 joblib requests

import json, os, datetime, warnings, re, time
import numpy as np
import pandas as pd
import requests
import shap
import joblib
import h3
import anthropic
from zoneinfo import ZoneInfo

from google.colab import auth, drive, userdata
import shutil

# Aviso benigno do SHAP para classificador binário LightGBM (saída vira lista de ndarray);
# o código trata esse formato em _top_shap, então silenciamos o ruído.
warnings.filterwarnings('ignore', message='.*binary classifier with TreeExplainer.*')

print('Imports OK')

## 2. Configuração


In [ ]:
# ---- Chaves (Colab Secrets) ----
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
try:
    OPENMETEO_API_KEY = userdata.get('OPENMETEO_API_KEY')   # chave comercial (opcional)
except Exception:
    OPENMETEO_API_KEY = None
MODEL_LLM = 'claude-haiku-4-5-20251001'   # rápido e econômico para a narração

# ---- Paths no Drive (contrato do 05_6) ----
DRIVE_BASE         = '/content/drive/MyDrive/DOUTORADO'
MODELO_CLIMA_PATH  = f'{DRIVE_BASE}/modelo_par_clima.joblib'
META_CLIMA_PATH    = f'{DRIVE_BASE}/modelo_par_clima_meta.json'
MODELO_ATRASO_PATH = f'{DRIVE_BASE}/modelo_par_atraso.joblib'
META_ATRASO_PATH   = f'{DRIVE_BASE}/modelo_par_atraso_meta.json'
BASE_DIR           = f'{DRIVE_BASE}/003_DADOS_SINTETICOS/arquivos_base'

# ---- Decisão ----
LIMIAR_EVENTO = 0.5     # prob >= limiar em qualquer dimensão => há demanda ("pico")
LIMIAR_BAIXA  = 0.45    # prob < este valor => chance BAIXA (a narração NÃO deve recomendar por ela)
HORAS_SCAN    = [1, 2]  # passo 6: varredura para frente se não houver demanda no horário pedido

# Critério de RECORRÊNCIA do atraso (feedback: só sugerir ir se o padrão for consistente).
# Um slot (hora×dia-da-semana) só é considerado "recorrente" se voos chegaram atrasados em
# pelo menos MIN_DIAS_RECORRENCIA datas distintas no histórico, com concentração relevante
# de uma mesma CIA/origem (>= CONC_MIN_RECORRENCIA da amostra do slot).
MIN_DIAS_RECORRENCIA  = 3
CONC_MIN_RECORRENCIA  = 0.40

# ---- Log das consultas ao LLM ----
LOG_NARRADOR = f'{DRIVE_BASE}/log_narrador.jsonl'
# Preço do Haiku 4.5 (USD por 1M tokens) — usado só para estimar custo no log.
PRECO_LLM = {'input': 1.00, 'output': 5.00, 'cache_read': 0.10, 'cache_write': 1.25}

# ---- Fuso: modelos treinados em UTC; motorista fala em horário local ----
TZ_LOCAL = ZoneInfo('America/Sao_Paulo')

# ---- Aeroporto Salgado Filho (SBPA) ----
AER_LAT, AER_LNG = -29.9939, -51.1711

# ---- Distância representativa (contexto do motorista) ----
# Os modelos usam dist_aeroporto_km (origem->SBPA) por EVENTO. A pergunta do narrador é sobre
# DEMANDA no horário (não um evento específico), então mantemos a distância FIXA num valor
# urbano representativo e a tratamos como variável de contexto constante (documentado). Ajuste
# se quiser simular o motorista já próximo (~0) ou mais distante do aeroporto.
DIST_REPRESENTATIVA_KM = 8.0

# ---- Endpoints Open-Meteo (previsão futura) ----
OPENMETEO_URL = ('https://customer-api.open-meteo.com/v1/forecast' if OPENMETEO_API_KEY
                 else 'https://api.open-meteo.com/v1/forecast')

print('Config OK | limiar =', LIMIAR_EVENTO, '| baixa <', LIMIAR_BAIXA,
      '| Open-Meteo com chave:', bool(OPENMETEO_API_KEY))

## 3. Autenticação e carga dos dois modelos pareados


In [ ]:
auth.authenticate_user()
drive.mount('/content/drive')

def _carrega_par(model_path, meta_path):
    m = joblib.load(model_path)
    with open(meta_path, encoding='utf-8') as f:
        meta = json.load(f)
    return m, meta

model_clima,  meta_clima  = _carrega_par(MODELO_CLIMA_PATH,  META_CLIMA_PATH)
model_atraso, meta_atraso = _carrega_par(MODELO_ATRASO_PATH, META_ATRASO_PATH)

# Janela e uso de distância vêm dos metadados (idênticos nos dois modelos).
JANELA   = meta_clima['janela']            # '1h' na produção
USE_DIST = bool(meta_clima['dist_aeroporto'])

# Registro central: cada modelo pareado usa as SUAS próprias features (binário evento vs OUTROS).
MODELOS = {
    'clima':  {'model': model_clima,  'features': meta_clima['features_lgbm'],  'meta': meta_clima},
    'atraso': {'model': model_atraso, 'features': meta_atraso['features_lgbm'], 'meta': meta_atraso},
}

print(f'Janela = {JANELA} | usa distância = {USE_DIST}')
for nome, d in MODELOS.items():
    m = d['meta']
    print(f"  modelo '{nome}': {len(d['features'])} features | "
          f"macro_f1_bal={m.get('macro_f1_bal')} | pr_auc={m.get('pr_auc')} | "
          f"posfato_atraso={m.get('posfato_atraso')}")


## 4. Dados de apoio + PERFIL histórico de voos (por hora × dia-da-semana)

O CSV `03_voos_atrasados_sbpa.csv` contém **apenas voos atrasados**. Como não há API de voos
futuros, derivamos o **contexto típico de atrasos** por (hora UTC × dia-da-semana): quantos voos
atrasados, quantas empresas, atraso médio/máximo e a composição (CIA/linha/origem/severidade)
que ocorrem tipicamente naquele horário. É o melhor proxy do sinal de voos para uma data futura.


In [ ]:
shutil.copy(f'{BASE_DIR}/dados_meteorologicos_utci_horario.csv', './')
shutil.copy(f'{BASE_DIR}/DADOS_AEROPORTO/03_voos_atrasados_sbpa.csv', './')

df_clima = pd.read_csv('/content/dados_meteorologicos_utci_horario.csv')
df_voos  = pd.read_csv('/content/03_voos_atrasados_sbpa.csv', sep=';')

# UTC em tudo (como no 05_6): clima e voos.
df_clima['time'] = pd.to_datetime(df_clima['time'], utc=True, errors='coerce')
df_clima = df_clima.dropna(subset=['time']).sort_values('time').reset_index(drop=True)
df_voos['CHEGADA_REAL'] = pd.to_datetime(df_voos['CHEGADA_REAL'], errors='coerce', utc=True)
df_voos['time_window']  = df_voos['CHEGADA_REAL'].dt.floor(JANELA)

# Mesmas listas do 05_6 (as features enriquecidas de voo dependem delas).
TOP_CIAS    = ['TAM', 'AZU', 'GLO', 'TAP', 'ARG']
TOP_ORIGENS = ['SBGR', 'SBSP', 'SBKP', 'SBCF', 'SBBR']

def _agregar_voos(dfv):
    """Agregação ENRIQUECIDA de voos atrasados por janela (idêntica ao 05_6)."""
    dfv = dfv.dropna(subset=['time_window']).copy()
    if 'CODIGO_TIPO_LINHA' in dfv.columns:
        lt = dfv['CODIGO_TIPO_LINHA'].astype(str).str.strip().str.upper()
        dfv['linha_N'] = (lt == 'N').astype(int); dfv['linha_I'] = (lt == 'I').astype(int)
    if 'ICAO_EMPRESA_AEREA' in dfv.columns:
        cia = dfv['ICAO_EMPRESA_AEREA'].astype(str).str.upper()
        for c in TOP_CIAS: dfv[f'cia_{c}'] = (cia == c).astype(int)
    if 'ICAO_AERODROMO_ORIGEM' in dfv.columns:
        org = dfv['ICAO_AERODROMO_ORIGEM'].astype(str).str.upper()
        for o in TOP_ORIGENS: dfv[f'orig_{o}'] = (org == o).astype(int)
    if 'STATUS_ATRASO' in dfv.columns:
        st = dfv['STATUS_ATRASO'].astype(str)
        dfv['sev_leve']     = st.str.contains('Leve', case=False, na=False).astype(int)
        dfv['sev_moderado'] = st.str.contains('Moderado', case=False, na=False).astype(int)
        dfv['sev_grave']    = st.str.contains('Grave', case=False, na=False).astype(int)
    agg = {'qtd_voos_previstos': ('time_window', 'size')}
    if 'ICAO_EMPRESA_AEREA' in dfv.columns:
        agg['qtd_empresas_aereas'] = ('ICAO_EMPRESA_AEREA', 'nunique')
    if 'ATRASO_MINUTOS' in dfv.columns:
        dfv['ATRASO_MINUTOS'] = pd.to_numeric(dfv['ATRASO_MINUTOS'], errors='coerce')
        agg['atraso_medio_min'] = ('ATRASO_MINUTOS', 'mean')
        agg['atraso_max_min']   = ('ATRASO_MINUTOS', 'max')
    ind_cols = [c for c in dfv.columns if c.startswith(('linha_', 'cia_', 'orig_', 'sev_'))]
    for c in ind_cols:
        agg[f'voos_{c}'] = (c, 'sum')
    return dfv.groupby('time_window').agg(**agg).reset_index()

voos_win = _agregar_voos(df_voos)
COLS_VOOS = [c for c in voos_win.columns if c != 'time_window']

# PERFIL MARGINAL por (hora UTC, dia-da-semana UTC): reindexado sobre TODAS as janelas do
# período (as janelas SEM voo atrasado entram como 0), reproduzindo a distribuição com zeros
# que o 05_6 viu no treino (voos_agg é mergeado how='left' e as janelas ausentes recebem
# fillna(0)). Usar a média só das janelas com atraso (média CONDICIONAL) superestimaria o sinal
# típico de voo e enviesaria o veredito para SIM. A grade completa vem do CSV de clima (horário).
_calendario = pd.DataFrame({'time_window': pd.Index(df_clima['time'].dt.floor(JANELA).unique())})
_voos_marg = _calendario.merge(voos_win, on='time_window', how='left')
for c in COLS_VOOS:
    _voos_marg[c] = _voos_marg[c].fillna(0.0)
_voos_marg['hora']       = _voos_marg['time_window'].dt.hour
_voos_marg['dia_semana'] = _voos_marg['time_window'].dt.dayofweek
PERFIL_VOOS = _voos_marg.groupby(['hora', 'dia_semana'])[COLS_VOOS].mean()

# Climatologia de fallback do clima (média por hora×mês), caso a API falhe e não haja histórico.
COLS_CLIMA_CSV = [c for c in df_clima.select_dtypes('number').columns]
_clima_hm = df_clima.copy()
_clima_hm['hora'] = _clima_hm['time'].dt.hour; _clima_hm['mes'] = _clima_hm['time'].dt.month
CLIMATOLOGIA = _clima_hm.groupby(['hora', 'mes'])[COLS_CLIMA_CSV].mean()

print(f'Clima: {df_clima.shape} | Voos: {df_voos.shape}')
print(f'Features de voo: {COLS_VOOS}')
print(f'Perfil de voos: {PERFIL_VOOS.shape[0]} combinações (hora×dia-semana).')


## 5. Clima futuro via Open-Meteo (previsão)

Buscamos apenas as variáveis climáticas que **os modelos realmente usam** (derivadas de
`features_lgbm`, incluindo as bases de `lag`/`diff`). Fallback em cascata: **API → histórico do
CSV (mesma hora) → climatologia (média hora×mês)**.


In [ ]:
# Variáveis horárias que a Open-Meteo oferece (whitelist p/ montar o pedido a partir das features).
# IMPORTANTE: o CSV dados_meteorologicos_utci_horario.csv foi construído a partir da Open-Meteo,
# então incluímos TODAS as variáveis horárias que ela expõe e que podem aparecer como coluna crua
# no CSV (radiação, weather_code, is_day, sunshine_duration, ...). Assim, qualquer feature física
# que o 05_6 tenha selecionado é reproduzível aqui e NÃO vira NaN constante em produção (o único
# grupo climático não-reproduzível seriam índices UTCI/compostos — mas o 05_6 os EXCLUI via
# eh_score_composto, que casa a substring 'utci', então não entram em features_lgbm).
OPENMETEO_DISPONIVEIS = {
    'temperature_2m', 'relative_humidity_2m', 'dew_point_2m', 'apparent_temperature',
    'precipitation', 'rain', 'showers', 'snowfall', 'snow_depth', 'surface_pressure', 'pressure_msl',
    'cloud_cover', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high',
    'wind_speed_10m', 'wind_speed_80m', 'wind_speed_100m', 'wind_speed_120m', 'wind_speed_180m',
    'wind_gusts_10m', 'wind_direction_10m', 'wind_direction_100m',
    'visibility', 'et0_fao_evapotranspiration', 'vapour_pressure_deficit',
    'weather_code', 'is_day', 'sunshine_duration', 'cape', 'freezing_level_height',
    'shortwave_radiation', 'direct_radiation', 'diffuse_radiation',
    'direct_normal_irradiance', 'global_tilted_irradiance', 'terrestrial_radiation',
    'shortwave_radiation_instant', 'direct_radiation_instant', 'diffuse_radiation_instant',
    'soil_temperature_0cm', 'soil_temperature_6cm', 'soil_moisture_0_to_1cm',
}

def _base_clima(feat):
    """Nome da variável física por trás de uma feature (remove prefixo diff_/sufixo _lag)."""
    if feat.startswith('diff_'):
        return feat[len('diff_'):]
    if feat.endswith('_lag'):
        return feat[:-len('_lag')]
    return feat

# União das variáveis climáticas necessárias pelos dois modelos ∩ disponíveis na Open-Meteo.
VARS_CLIMA = sorted({
    _base_clima(f) for d in MODELOS.values() for f in d['features']
    if _base_clima(f) in OPENMETEO_DISPONIVEIS
})
if not VARS_CLIMA:   # segurança: pelo menos as 4 do contrato
    VARS_CLIMA = ['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'surface_pressure']
print('Variáveis climáticas buscadas na Open-Meteo:', VARS_CLIMA)

# --- Diagnóstico de reprodutibilidade (evita a "falsa garantia" de compatibilidade por nome) ---
# Sinaliza qualquer feature dos modelos que NÃO seja temporal, de voo, distância nem clima
# reproduzível pela Open-Meteo — essas seriam servidas como NaN constante (skew treino×inferência).
_TEMPORAIS = {'hora', 'mes', 'dia_semana', 'hora_sin', 'hora_cos', 'mes_sin', 'mes_cos',
              'dia_semana_sin', 'dia_semana_cos'}
def _eh_voo(f):
    return (f.startswith('voos_') or
            f in ('qtd_voos_previstos', 'qtd_empresas_aereas', 'atraso_medio_min', 'atraso_max_min'))
_nao_reproduziveis = sorted({
    f for d in MODELOS.values() for f in d['features']
    if f not in _TEMPORAIS and not _eh_voo(f) and f != 'dist_aeroporto_km'
    and _base_clima(f) not in OPENMETEO_DISPONIVEIS
})
if _nao_reproduziveis:
    print('ATENÇÃO: features do modelo que a Open-Meteo NÃO reproduz (serão NaN em produção):')
    print('  ', _nao_reproduziveis)
    print('  -> considere adicioná-las a OPENMETEO_DISPONIVEIS (se a Open-Meteo as oferece) '
          'ou excluí-las no 05_6 para garantir paridade treino×inferência.')
else:
    print('OK: toda feature climática dos modelos é reproduzível pela Open-Meteo.')

# Alguma feature de voo entrou nos modelos? Depende de INCLUIR_FEATURES_POSFATO_ATRASO no 05_6.
# Sob o padrão RIGOROSO (False), o CSV de voos é só-atrasados (pós-fato) e as features de voo ficam
# FORA de ambos os modelos: a dimensão "atraso" prevê corridas geradas por atraso a partir de
# padrões de CLIMA+TEMPO, sem citar contagens/atrasos diretos. Isso muda o que a narração pode afirmar.
FEATURES_VOO_ATIVAS = any(_eh_voo(f) for d in MODELOS.values() for f in d['features'])
if FEATURES_VOO_ATIVAS:
    print('Features de voo ATIVAS: a narração pode citar contagens/atrasos de voo diretamente.')
else:
    print('Features de voo INATIVAS (padrão rigoroso pós-fato): a dimensão "atraso" usa só '
          'clima+tempo. Rode o 05_6 com INCLUIR_FEATURES_POSFATO_ATRASO=True para citar '
          'valores diretos de atraso (aceitando o caveat de pós-fato).')

_CLIMA_CACHE = {}   # Timestamp UTC (floor h) -> {var: valor}

def _buscar_openmeteo(dt_ini_utc, dt_fim_utc):
    """Preenche _CLIMA_CACHE com a previsão horária (UTC) no intervalo pedido. True se OK."""
    params = {
        'latitude': AER_LAT, 'longitude': AER_LNG,
        'hourly': ','.join(VARS_CLIMA), 'timezone': 'UTC',
        'start_date': pd.Timestamp(dt_ini_utc).date().isoformat(),
        'end_date':   pd.Timestamp(dt_fim_utc).date().isoformat(),
        'wind_speed_unit': 'kmh', 'temperature_unit': 'celsius',
    }
    if OPENMETEO_API_KEY:
        params['apikey'] = OPENMETEO_API_KEY
    try:
        r = requests.get(OPENMETEO_URL, params=params, timeout=30)
        r.raise_for_status()
        h = r.json()['hourly']
        tempos = pd.to_datetime(h['time'], utc=True)
        for i, t in enumerate(tempos):
            tw = pd.Timestamp(t).floor('h')
            _CLIMA_CACHE[tw] = {v: (float(h[v][i]) if h.get(v) and h[v][i] is not None else np.nan)
                                for v in VARS_CLIMA}
        return True
    except Exception as e:
        print(f'[Open-Meteo] indisponível ({e}); usando fallback histórico/climatologia.')
        return False

def _clima_na_hora(ts_utc):
    """Retorna dict {var: valor} para a hora UTC pedida (API → CSV → climatologia)."""
    tw = pd.Timestamp(ts_utc).floor('h')
    if tw not in _CLIMA_CACHE:
        _buscar_openmeteo(tw - pd.Timedelta('1h'), tw + pd.Timedelta(f'{max(HORAS_SCAN)}h'))
    if tw in _CLIMA_CACHE:
        return _CLIMA_CACHE[tw]
    # fallback 1: histórico exato do CSV
    linha = df_clima[df_clima['time'] == tw]
    if not linha.empty:
        return {v: float(linha[v].values[0]) for v in VARS_CLIMA if v in linha.columns}
    # fallback 2: climatologia (média por hora×mês)
    chave = (tw.hour, tw.month)
    if chave in CLIMATOLOGIA.index:
        row = CLIMATOLOGIA.loc[chave]
        return {v: float(row[v]) for v in VARS_CLIMA if v in CLIMATOLOGIA.columns}
    return {}   # nada disponível -> features de clima ficam NaN (o LightGBM trata)

print('Busca de clima pronta (_clima_na_hora).')


## 6. Engenharia de features de uma janela (UTC-aware)

`construir_features(dt_local)` recebe a hora **local** do motorista, converte para **UTC**, busca
clima (Open-Meteo) + perfil de voos e monta o superconjunto de features no namespace do `05_6`.


In [ ]:
def _para_utc(dt_local):
    """datetime/Timestamp local (naive ou tz) -> Timestamp UTC."""
    t = pd.Timestamp(dt_local)
    t = t.tz_localize(TZ_LOCAL) if t.tzinfo is None else t.tz_convert(TZ_LOCAL)
    return t.tz_convert('UTC')

def _eh_feature_voo(f):
    return (f.startswith('voos_') or
            f in ('qtd_voos_previstos', 'qtd_empresas_aereas', 'atraso_medio_min', 'atraso_max_min'))

def construir_features(dt_local):
    """Retorna (DataFrame 1 linha com superconjunto de features, dict de contexto)."""
    t_utc = _para_utc(dt_local).floor('h')          # JANELA=1h => floor horário
    clima_atual = _clima_na_hora(t_utc)
    clima_lag   = _clima_na_hora(t_utc - pd.Timedelta('1h'))

    row = {}
    # Clima na janela + lags/diffs (mantém NaN quando ausente; NÃO preenche com 0).
    for v in VARS_CLIMA:
        row[v] = clima_atual.get(v, np.nan)
        row[f'{v}_lag'] = clima_lag.get(v, np.nan)
        row[f'diff_{v}'] = (row[v] - row[f'{v}_lag']
                            if pd.notna(row[v]) and pd.notna(row[f'{v}_lag']) else np.nan)

    # Perfil típico de voos por (hora UTC, dia-da-semana UTC).
    chave = (t_utc.hour, t_utc.dayofweek)
    if chave in PERFIL_VOOS.index:
        prow = PERFIL_VOOS.loc[chave]
        for c in COLS_VOOS:
            row[c] = float(prow[c])
    else:
        for c in COLS_VOOS:
            row[c] = 0.0

    # Distância (contexto fixo) — só relevante se USE_DIST e a feature estiver no modelo.
    row['dist_aeroporto_km'] = DIST_REPRESENTATIVA_KM

    # Temporal (UTC, como no treino).
    row['hora'] = t_utc.hour; row['mes'] = t_utc.month; row['dia_semana'] = t_utc.dayofweek
    for col, period in [('hora', 24), ('mes', 12), ('dia_semana', 7)]:
        row[f'{col}_sin'] = np.sin(2*np.pi * row[col] / period)
        row[f'{col}_cos'] = np.cos(2*np.pi * row[col] / period)

    dt_loc = _para_utc(dt_local).tz_convert(TZ_LOCAL)
    ctx = {'t_utc': t_utc, 'dt_local': dt_loc,
           'clima_disponivel': bool(clima_atual)}
    return pd.DataFrame([row]), ctx

def _preparar_X(df_full, features):
    """Recorta o superconjunto para as features do modelo. Consistência com o 05_6:
    features de VOO ausentes -> 0; clima/lag/diff/temporal ausentes -> NaN (LightGBM trata)."""
    d = df_full.copy()
    for f in features:
        if f not in d.columns:
            d[f] = 0.0 if _eh_feature_voo(f) else np.nan
    X = d[features].copy()
    for f in features:                       # só voos recebem fillna(0), como no treino
        if _eh_feature_voo(f):
            X[f] = X[f].fillna(0.0)
    return X

print('construir_features() e _preparar_X() prontos (UTC-aware; NaN preservado no clima).')


## 7. Predição + SHAP (o que o modelo "vê" nas condições)


In [ ]:
explainers = {nome: shap.TreeExplainer(d['model']) for nome, d in MODELOS.items()}

DIAS_SEMANA = ['segunda', 'terça', 'quarta', 'quinta', 'sexta', 'sábado', 'domingo']

_CIA_NOMES = {
    'TAM': 'LATAM', 'AZU': 'Azul', 'GLO': 'Gol', 'TAP': 'TAP', 'ARG': 'Aerolíneas Argentinas',
}
_ORIG_NOMES = {
    'SBGR': 'Guarulhos (SP)', 'SBSP': 'Congonhas (SP)',
    'SBKP': 'Viracopos (SP)', 'SBCF': 'Confins (MG)', 'SBBR': 'Brasília (DF)',
}

def _fmt_valor(v):
    return 'ausente (sem leitura)' if (v is None or (isinstance(v, float) and np.isnan(v))) else v

def _qtd_voos_coloquial(v):
    """Contagem média de voos que CHEGAM atrasados -> texto coloquial para o motorista.
    Enquadramento: o motorista busca passageiros que DESEMBARCAM de voos atrasados na chegada."""
    if v < 0.1:  return 'sem histórico de voos chegando atrasados nesse horário'
    if v < 0.6:  return 'raramente chega 1 voo atrasado (pouco desembarque extra)'
    if v < 1.4:  return 'cerca de 1 voo chegando atrasado'
    if v < 2.5:  return 'cerca de 2 voos chegando atrasados'
    if v < 3.5:  return 'cerca de 3 voos chegando atrasados'
    return f'em torno de {int(round(v))} voos chegando atrasados'

def _label_feature(feat, valor):
    """Nome técnico -> descrição legível com o VALOR real (para o motorista conferir)."""
    v = _fmt_valor(valor)
    if isinstance(v, str):
        base = {'temperature_2m': 'temperatura', 'relative_humidity_2m': 'umidade relativa',
                'wind_speed_10m': 'velocidade do vento', 'surface_pressure': 'pressão'}.get(feat, feat)
        return f'{base} ({v})'
    mapa = {
        'dist_aeroporto_km':        f'distância ao aeroporto ({v:.1f} km)',
        'temperature_2m':           f'temperatura ({v:.1f}°C)',
        'relative_humidity_2m':     f'umidade relativa ({v:.0f}%)',
        'dew_point_2m':             f'ponto de orvalho ({v:.1f}°C)',
        'apparent_temperature':     f'sensação térmica ({v:.1f}°C)',
        'wind_speed_10m':           f'velocidade do vento ({v:.1f} km/h)',
        'wind_gusts_10m':           f'rajadas de vento ({v:.1f} km/h)',
        'wind_direction_10m':       f'direção do vento ({v:.0f}°)',
        'precipitation':            f'precipitação ({v:.1f} mm)',
        'rain':                     f'chuva ({v:.1f} mm)',
        'surface_pressure':         f'pressão à superfície ({v:.0f} hPa)',
        'pressure_msl':             f'pressão ao nível do mar ({v:.0f} hPa)',
        'cloud_cover':              f'cobertura de nuvens ({v:.0f}%)',
        'vapour_pressure_deficit':  f'déficit de pressão de vapor ({v:.2f} kPa)',
        'qtd_voos_previstos':       _qtd_voos_coloquial(v),
        'qtd_empresas_aereas':      f'{v:.0f} empresa(s) aérea(s) com chegada atrasada típica',
        'atraso_medio_min':         f'atraso médio histórico na chegada de {v:.0f} min',
        'atraso_max_min':           f'atraso máximo histórico na chegada de {v:.0f} min',
        'hora':                     f'hora do dia UTC ({int(v)}h)',
        'mes':                      f'mês ({int(v)})',
        'dia_semana':               f'dia da semana UTC ({DIAS_SEMANA[int(v) % 7]})',
    }
    if feat in mapa:
        return mapa[feat]
    if feat.startswith('diff_'):
        return f'variação de {feat[len("diff_"):]} ({v:+.2f})'
    if feat.endswith('_lag'):
        return f'{feat[:-4]} na hora anterior ({v:.2f})'
    if feat.startswith('voos_cia_'):
        cia = feat[len('voos_cia_'):]
        return f'voos da {_CIA_NOMES.get(cia, cia)} chegando atrasados ({v:.1f})'
    if feat.startswith('voos_orig_'):
        orig = feat[len('voos_orig_'):]
        return f'voos atrasados chegando de {_ORIG_NOMES.get(orig, orig)} ({v:.1f})'
    if feat == 'voos_linha_N':         return f'voos nacionais chegando atrasados ({v:.1f})'
    if feat == 'voos_linha_I':         return f'voos internacionais chegando atrasados ({v:.1f})'
    if feat.startswith('voos_sev_'):   return f'chegadas com atraso {feat[len("voos_sev_"):]} ({v:.1f})'
    if feat.endswith(('_sin', '_cos')): return f'{feat} (componente cíclica)'
    return f'{feat} ({v:.3f})'

def _detalhe_voos_historicos(t_utc):
    """Perfil histórico de voos que CHEGAM atrasados no slot (hora UTC × dia-da-semana).
    Enquadramento correto: são passageiros DESEMBARCANDO na chegada — é a eles que o motorista
    atende. Retorna dict com texto legível + métricas de RECORRÊNCIA, ou None se não houver slot.

    Recorrência: exige atrasos em >= MIN_DIAS_RECORRENCIA datas distintas E concentração de uma
    mesma CIA ou origem >= CONC_MIN_RECORRENCIA. Sem isso, o padrão é esporádico (não recomendar)."""
    mask = ((df_voos['CHEGADA_REAL'].dt.hour == t_utc.hour) &
            (df_voos['CHEGADA_REAL'].dt.dayofweek == t_utc.dayofweek))
    slot = df_voos[mask].dropna(subset=['CHEGADA_REAL'])
    if slot.empty:
        return None

    n_registros = int(len(slot))
    n_dias = int(slot['CHEGADA_REAL'].dt.normalize().nunique())

    partes = []
    conc_cia = conc_orig = 0.0
    if 'ICAO_EMPRESA_AEREA' in slot.columns:
        vc = slot['ICAO_EMPRESA_AEREA'].value_counts()
        conc_cia = float(vc.iloc[0] / n_registros) if not vc.empty else 0.0
        nomes = [_CIA_NOMES.get(c, c) for c in vc.head(2).index.tolist()]
        partes.append(f'CIA: {" / ".join(nomes)}')
    if 'ICAO_AERODROMO_ORIGEM' in slot.columns:
        vc = slot['ICAO_AERODROMO_ORIGEM'].value_counts()
        conc_orig = float(vc.iloc[0] / n_registros) if not vc.empty else 0.0
        nomes = [_ORIG_NOMES.get(o, o) for o in vc.head(2).index.tolist()]
        partes.append(f'Chegando de: {" / ".join(nomes)}')
    if 'CODIGO_TIPO_LINHA' in slot.columns:
        lt = slot['CODIGO_TIPO_LINHA'].astype(str).str.strip().str.upper().value_counts()
        if not lt.empty:
            dom = lt.index[0]
            partes.append('Linha: ' + ('nacional' if dom == 'N' else 'internacional' if dom == 'I' else dom))
    t_local = t_utc.tz_convert(TZ_LOCAL)
    med_min = int(slot['CHEGADA_REAL'].dt.minute.median())
    partes.append(f'Chegada (desembarque) prevista: por volta das {t_local.hour:02d}h{med_min:02d} (horário local)')
    partes.append(f'Histórico: {n_registros} chegadas atrasadas em {n_dias} dia(s) distinto(s)')

    recorrente = (n_dias >= MIN_DIAS_RECORRENCIA and
                  max(conc_cia, conc_orig) >= CONC_MIN_RECORRENCIA)
    return {'texto': ' | '.join(partes), 'recorrente': recorrente,
            'n_dias': n_dias, 'n_registros': n_registros,
            'conc_cia': conc_cia, 'conc_orig': conc_orig}

def _top_shap(nome, X, top_n):
    """Top-N features por |SHAP| para a classe positiva do modelo binário `nome`."""
    sv = explainers[nome].shap_values(X)
    if isinstance(sv, list):
        vals = sv[1][0]
    elif getattr(sv, 'ndim', 2) == 3:
        vals = sv[0, :, 1]
    else:
        vals = sv[0]
    feats = MODELOS[nome]['features']; fv = X.values[0]
    ordem = np.argsort(np.abs(vals))[::-1][:top_n]
    return [{'feature': feats[i], 'valor': float(fv[i]) if pd.notna(fv[i]) else None,
             'shap': float(vals[i]),
             'label': _label_feature(feats[i], fv[i])} for i in ordem]

def predizer(dt_local, top_n=6):
    """Roda os dois modelos pareados sobre a janela e decide se há demanda ('pico')."""
    df_full, ctx = construir_features(dt_local)
    dims = {}
    for nome, d in MODELOS.items():
        X = _preparar_X(df_full, d['features'])
        prob = float(d['model'].predict_proba(X)[0][1])
        dims[nome] = {'prob': prob, 'top_shap': _top_shap(nome, X, top_n)}
    p_clima, p_atraso = dims['clima']['prob'], dims['atraso']['prob']
    tem_demanda = (p_clima >= LIMIAR_EVENTO) or (p_atraso >= LIMIAR_EVENTO)
    if tem_demanda:
        dim_dom = 'clima' if p_clima >= p_atraso else 'atraso'
    else:
        dim_dom = None
    return {
        'tem_demanda': tem_demanda,
        'dimensao_dominante': dim_dom,
        'prob_clima': p_clima, 'prob_atraso': p_atraso,
        'clima': dims['clima'], 'atraso': dims['atraso'],
        'contexto': ctx,
    }

print('predizer() pronto (decide demanda e retorna SHAP das duas dimensões).')

## 8. Ficha técnica (features + valores que decidiram)

Passo 5 do fluxo: quando há demanda, expomos **exatamente quais features e com quais valores**
puxaram a decisão — para o motorista conferir pessoalmente e reportar se acertou.


In [ ]:
def ficha_tecnica(res, dimensao=None, top_n=6):
    """Imprime a ficha objetiva feature -> valor -> efeito da dimensão indicada
    (ou da dominante, se houver demanda; senão, da de maior probabilidade)."""
    dim = dimensao or res['dimensao_dominante'] or ('clima' if res['prob_clima'] >= res['prob_atraso'] else 'atraso')
    rotulo = {'clima': 'EVENTO CLIMÁTICO', 'atraso': 'ATRASO DE VOO'}[dim]
    print(f'  Ficha técnica — dimensão {rotulo} (prob {res[f"prob_{dim}"]:.1%}):')
    for t in res[dim]['top_shap'][:top_n]:
        efeito = 'AUMENTA' if t['shap'] > 0 else 'reduz'
        print(f'    • {t["label"]:<48} {efeito} a chance  [SHAP {t["shap"]:+.3f}]')
    return dim

print('ficha_tecnica() pronta.')


## 9. Varredura +2h (passo 6)

Se não há demanda no horário pedido, procuramos nas próximas horas (`HORAS_SCAN = [1, 2]`).


In [ ]:
def varrer_para_frente(dt_local, horas=None):
    """Avalia dt_local + h para cada h em `horas`; devolve o primeiro com demanda (ou None)."""
    horas = horas or HORAS_SCAN
    achados = []
    for h in horas:
        alvo = pd.Timestamp(dt_local) + pd.Timedelta(hours=h)
        r = predizer(alvo)
        achados.append({'horas_a_frente': h, 'dt_local': alvo, 'resultado': r})
        if r['tem_demanda']:
            return {'encontrou': True, 'slot': achados[-1], 'todos': achados}
    return {'encontrou': False, 'slot': None, 'todos': achados}

print('varrer_para_frente() pronto.')


## 10. Narração em linguagem natural (Claude, ancorada no SHAP)


In [ ]:
os.environ.pop('ANTHROPIC_WORKSPACE_ID', None)
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

_INSTR_VOO = (
    'SEMPRE cite os VALORES concretos que pesaram (ex.: "temperatura de 33°C", '
    '"atraso médio de 40 min na chegada"). Quando o contexto histórico de voos estiver '
    'disponível (CIA, de onde chegam, linha, horário de chegada), incorpore esses detalhes '
    'naturalmente — o motorista quer saber quais desembarques esperar.'
    if FEATURES_VOO_ATIVAS else
    'Cite os VALORES climáticos/temporais concretos que pesaram (ex.: "temperatura de 33°C", '
    '"18h de sexta"). NÃO invente números de voos nem minutos de atraso: nesta configuração '
    'a leitura de atraso de voo vem de PADRÕES de clima e horário, não de contagens diretas. '
    'Quando o contexto histórico de voos estiver disponível (CIA, de onde chegam, linha, '
    'horário de chegada), use esses dados para contextualizar — mas sem alegar que são previstos.')

SYSTEM_PROMPT = f"""Você é um assistente de demanda de corridas para motoristas de aplicativo em Porto Alegre, focado no Aeroporto Salgado Filho (SBPA).
Explique a previsão de forma clara, direta e prática, em português, em 3 a 5 frases curtas.
NÃO use jargão técnico (nada de "SHAP", "modelo", "probabilidade" cru, "feature").
{_INSTR_VOO}

ENQUADRAMENTO OBRIGATÓRIO — o motorista busca passageiros que CHEGAM (desembarcam) de voos atrasados.
NUNCA fale em voos que "saem" ou "partem" atrasados: o que gera corrida é o DESEMBARQUE na chegada.

SEJA CONSERVADOR. Não é otimista por padrão. Regras de decisão:
- Chance abaixo de 45% é BAIXA: por si só NÃO justifica ir ao aeroporto.
- Só recomende IR quando houver sinal forte E consistente: uma das dimensões com chance claramente
  alta (>= 50%) e, no caso de atraso de voo, um padrão RECORRENTE (o histórico deve indicar atrasos
  na chegada se repetindo em vários dias, concentrados numa mesma CIA/origem). Se o padrão for
  esporádico ou fraco, recomende NÃO ir, mesmo que exista algum sinal.
- Na dúvida ou com sinais fracos/mistos, recomende NÃO se deslocar e explique o porquê.
Seja honesto sobre a incerteza. Termine com uma recomendação objetiva (ir ou não), coerente com essas regras."""

def _nivel_prob(p):
    if p < LIMIAR_BAIXA: return 'BAIXA'
    if p < LIMIAR_EVENTO: return 'MODERADA'
    if p < 0.65: return 'ALTA'
    return 'MUITO ALTA'

def _bloco_fatores(top):
    return '\n'.join(
        f'  - {t["label"]} — {"puxa a favor" if t["shap"] > 0 else "puxa contra"}'
        for t in top)

def _montar_prompt(res, cenario):
    ctx = res['contexto']
    quando = ctx['dt_local'].strftime('%d/%m/%Y às %Hh')
    dia = DIAS_SEMANA[ctx['dt_local'].dayofweek]
    aviso_clima = ('' if ctx['clima_disponivel']
                   else '\n(OBS: sem previsão climática para esta hora; baseado em histórico/climatologia.)')
    rotulo_atraso = ('ATRASO DE VOO na chegada (SBPA)' if FEATURES_VOO_ATIVAS
                     else 'ATRASO DE VOO na chegada (inferido por padrões de clima+horário; sem contagens diretas)')

    det = _detalhe_voos_historicos(ctx['t_utc'])
    if det is None:
        bloco_voos = ('\nContexto histórico de voos: sem registros de chegadas atrasadas nesse '
                      'horário e dia da semana.')
        recorrencia = 'RECORRÊNCIA do atraso: sem histórico — trate como sinal FRACO.'
    else:
        bloco_voos = ('\nContexto histórico de voos atrasados chegando nesse horário/dia da semana '
                      '(passageiros desembarcando):\n  ' + det['texto'])
        if det['recorrente']:
            recorrencia = (f'RECORRÊNCIA do atraso: RECORRENTE (atrasos na chegada em {det["n_dias"]} '
                           f'dias distintos, concentração de CIA/origem até '
                           f'{max(det["conc_cia"], det["conc_orig"]):.0%}). Padrão consistente.')
        else:
            recorrencia = (f'RECORRÊNCIA do atraso: NÃO recorrente / esporádico (só {det["n_dias"]} '
                           f'dia(s) distinto(s), concentração até '
                           f'{max(det["conc_cia"], det["conc_orig"]):.0%}). NÃO recomende ir só por '
                           f'causa do atraso.')

    return f"""Consulta do motorista para {quando} ({dia}).{aviso_clima}

Duas dimensões avaliadas de forma independente (cada uma contra "sem pico de corridas"):
  - Chance de pico por EVENTO CLIMÁTICO: {res['prob_clima']:.1%} (nível: {_nivel_prob(res['prob_clima'])})
  - Chance de pico por {rotulo_atraso}: {res['prob_atraso']:.1%} (nível: {_nivel_prob(res['prob_atraso'])})
Lembrete: níveis BAIXA (<45%) não justificam ir por si sós.

{recorrencia}

Fatores que mais pesaram na dimensão climática (com valores):
{_bloco_fatores(res['clima']['top_shap'])}

Fatores que mais pesaram na dimensão de atraso de voo na chegada (com valores):
{_bloco_fatores(res['atraso']['top_shap'])}
{bloco_voos}

Situação: {cenario}

Explique ao motorista, citando os valores concretos e os detalhes do voo quando disponíveis, se
vale a pena ir ao aeroporto e por quê. Siga as regras conservadoras: só recomende ir com sinal
forte e (para atraso) recorrente; caso contrário, recomende não ir."""

def _features_enviadas(res):
    """Features/valores efetivamente enviados ao prompt (para o log)."""
    def _lst(dim):
        return [{'feature': t['feature'], 'valor': t['valor'], 'shap': t['shap'], 'label': t['label']}
                for t in res[dim]['top_shap']]
    det = _detalhe_voos_historicos(res['contexto']['t_utc'])
    return {
        'prob_clima': res['prob_clima'], 'prob_atraso': res['prob_atraso'],
        'nivel_clima': _nivel_prob(res['prob_clima']), 'nivel_atraso': _nivel_prob(res['prob_atraso']),
        'clima': _lst('clima'), 'atraso': _lst('atraso'),
        'voos_historico': det,
    }

def _custo_tokens(usage):
    it = getattr(usage, 'input_tokens', 0) or 0
    ot = getattr(usage, 'output_tokens', 0) or 0
    cr = getattr(usage, 'cache_read_input_tokens', 0) or 0
    cw = getattr(usage, 'cache_creation_input_tokens', 0) or 0
    return round(it/1e6*PRECO_LLM['input'] + ot/1e6*PRECO_LLM['output']
                 + cr/1e6*PRECO_LLM['cache_read'] + cw/1e6*PRECO_LLM['cache_write'], 6)

def narrar(res, cenario):
    prompt = _montar_prompt(res, cenario)
    ts_req = datetime.datetime.now(TZ_LOCAL).isoformat()
    t0 = time.perf_counter()
    resp = client.messages.create(
        model=MODEL_LLM, max_tokens=400, system=SYSTEM_PROMPT,
        messages=[{'role': 'user', 'content': prompt}],
        workspace_id=anthropic.NOT_GIVEN)
    tempo_resp = round(time.perf_counter() - t0, 3)
    texto = resp.content[0].text

    registro = {
        'horario_solicitacao': ts_req,          # 4 - horário da solicitação (local)
        'tempo_resposta_s': tempo_resp,          # 5 - tempo de resposta
        'modelo': MODEL_LLM,
        'cenario': cenario,
        'features_enviadas': _features_enviadas(res),  # 1 - features e valores enviados ao prompt
        'prompt': prompt,                        # 2 - o prompt
        'resposta': texto,                       # 3 - a resposta
        'tokens': {                              # 6 - custo de tokens (contagem + USD)
            'input': getattr(resp.usage, 'input_tokens', None),
            'output': getattr(resp.usage, 'output_tokens', None),
            'cache_read': getattr(resp.usage, 'cache_read_input_tokens', None),
            'cache_write': getattr(resp.usage, 'cache_creation_input_tokens', None),
        },
        'custo_usd': _custo_tokens(resp.usage),
    }
    try:
        with open(LOG_NARRADOR, 'a', encoding='utf-8') as f:
            f.write(json.dumps(registro, ensure_ascii=False) + '\n')
    except Exception as e:
        print(f'[log] não consegui gravar em {LOG_NARRADOR}: {e}')
    return texto

print('narrar() pronto (conservador, enquadramento de chegada, com log em', LOG_NARRADOR, ').')

## 11. Ponto de entrada — pergunta do motorista → veredito

`consultar("amanhã às 18h")` executa o fluxo completo dos 6 passos.


In [ ]:
def _extrair_datetime_local(pergunta):
    """Extrai data+hora da pergunta (LLM) e devolve datetime LOCAL (naive, America/Sao_Paulo)."""
    agora = datetime.datetime.now(TZ_LOCAL)
    hoje = agora.strftime('%d/%m/%Y')
    resp = client.messages.create(
        model=MODEL_LLM, max_tokens=80,
        system=(f'Hoje é {hoje} (horário de Porto Alegre). Extraia a data e a hora mencionadas na '
                'pergunta e responda APENAS um JSON {"data":"DD/MM/YYYY","hora":HH}. '
                'Se não houver, use a próxima hora cheia.'),
        messages=[{'role': 'user', 'content': pergunta}],
        workspace_id=anthropic.NOT_GIVEN)
    try:
        bruto = resp.content[0].text.strip()
        m = re.search(r'\{.*\}', bruto, re.DOTALL)
        dados = json.loads(m.group(0) if m else bruto)
        return datetime.datetime.strptime(
            f"{dados['data']} {int(dados['hora']):02d}:00", '%d/%m/%Y %H:%M')
    except Exception:
        return agora.replace(minute=0, second=0, microsecond=0, tzinfo=None) + datetime.timedelta(hours=1)

def consultar(pergunta_ou_dt, imprimir=True):
    """Fluxo completo (6 passos). Aceita texto em linguagem natural ou datetime local."""
    if isinstance(pergunta_ou_dt, str):
        dt_local = _extrair_datetime_local(pergunta_ou_dt)
        pergunta = pergunta_ou_dt
    else:
        dt_local = pergunta_ou_dt
        pergunta = f'consulta para {pd.Timestamp(dt_local):%d/%m/%Y %Hh}'

    res = predizer(dt_local)
    quando = res['contexto']['dt_local'].strftime('%d/%m/%Y às %Hh')

    if res['tem_demanda']:
        cenario = (f'HÁ demanda esperada no horário pedido ({quando}); dimensão predominante: '
                   f'{"clima" if res["dimensao_dominante"]=="clima" else "atraso de voo"}.')
        narrativa = narrar(res, cenario)
        saida = {'pergunta': pergunta, 'dt_local': dt_local, 'veredito': 'SIM',
                 'resultado': res, 'narrativa': narrativa, 'scan': None}
    else:
        scan = varrer_para_frente(dt_local)
        if scan['encontrou']:
            s = scan['slot']; qs = s['resultado']['contexto']['dt_local'].strftime('%d/%m/%Y às %Hh')
            cenario = (f'NÃO há demanda no horário pedido ({quando}), MAS há em {qs} '
                       f'(+{s["horas_a_frente"]}h). Oriente o motorista a considerar esse horário.')
            narrativa = narrar(s['resultado'], cenario)
            saida = {'pergunta': pergunta, 'dt_local': dt_local, 'veredito': 'SIM (+adiante)',
                     'resultado': res, 'narrativa': narrativa, 'scan': scan}
        else:
            cenario = (f'NÃO há demanda no horário pedido ({quando}) nem nas próximas '
                       f'{max(HORAS_SCAN)} horas. Recomende não deslocar-se ao aeroporto por ora.')
            narrativa = narrar(res, cenario)
            saida = {'pergunta': pergunta, 'dt_local': dt_local, 'veredito': 'NÃO',
                     'resultado': res, 'narrativa': narrativa, 'scan': scan}

    if imprimir:
        print(f'Motorista : {pergunta}')
        print(f'Horário    : {quando}  (UTC {res["contexto"]["t_utc"]:%d/%m %Hh})')
        print(f'Veredito   : {saida["veredito"]}  '
              f'(clima {res["prob_clima"]:.1%} | atraso {res["prob_atraso"]:.1%})')
        print()
        # Ficha técnica da dimensão relevante (passo 5).
        alvo = saida['scan']['slot']['resultado'] if (saida['veredito'] == 'SIM (+adiante)') else res
        ficha_tecnica(alvo)
        print('\n--- Resposta ao motorista ---')
        print(saida['narrativa'])
        print()
    return saida

print('consultar() pronto. Use consultar("amanhã às 18h").')


## 12. Exemplos

Ajuste as perguntas/horários conforme necessário. Requer os modelos do `05_6` no Drive e as
chaves `ANTHROPIC_API_KEY` (e opcionalmente `OPENMETEO_API_KEY`) nos Colab Secrets.


In [ ]:
_ = consultar('Vale a pena ir ao aeroporto amanhã às 18h?')


In [ ]:
_ = consultar('E depois de amanhã de manhã, às 7h?')


In [ ]:
# Também aceita datetime local direto:
_ = consultar(datetime.datetime(2026, 9, 20, 15, 0, 0))